##Persiapan pustaka

In [1]:
%pip install requests beautifulsoup4 pandas 
import requests 
from bs4 import BeautifulSoup 
import pandas as pd
import time

from google_play_scraper import app, reviews, Sort

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


##Menyiapkan header request 

In [2]:
headers = {
    "User-Agent": "Mozilla/5.0 (compatible; DataMiningPracticum/1.0)",
    "Accept": "text/html,application/xhtml+xml",
    "Accept-Language": "en-US,en;q=0.9"
}

##Mengirim request dan memeriksa respons

In [3]:
url = "https://books.toscrape.com/"

response = requests.get(
    url,
    headers=headers,
    timeout=20
)

print("Status Code:", response.status_code)
print("URL:", response.url)
print("Respons HTML:")
print(response.text[:500])

Status Code: 200
URL: https://books.toscrape.com/
Respons HTML:
<!DOCTYPE html>
<!--[if lt IE 7]>      <html lang="en-us" class="no-js lt-ie9 lt-ie8 lt-ie7"> <![endif]-->
<!--[if IE 7]>         <html lang="en-us" class="no-js lt-ie9 lt-ie8"> <![endif]-->
<!--[if IE 8]>         <html lang="en-us" class="no-js lt-ie9"> <![endif]-->
<!--[if gt IE 8]><!--> <html lang="en-us" class="no-js"> <!--<![endif]-->
    <head>
        <title>
    All products | Books to Scrape - Sandbox
</title>

        <meta http-equiv="content-type" content="text/html; charset=UTF-8" /


##Membaca HTML dan menemukan elemen 

In [4]:
soup = BeautifulSoup(response.text, "html.parser")

print("Judul Website:")
print(soup.title.get_text(strip=True))

print("\nJumlah Buku pada Halaman Pertama:")
print(len(soup.select("article.product_pod")))

Judul Website:
All products | Books to Scrape - Sandbox

Jumlah Buku pada Halaman Pertama:
20


In [5]:

items = soup.select("article.product_pod")

item = items[0]

title = item.select_one("h3 a")["title"]

price = item.select_one(
    ".price_color"
).get_text(strip=True)

availability = item.select_one(
    ".availability"
).get_text(strip=True)

rating = item.select_one(
    "p.star-rating"
)["class"][1]

detail_url = item.select_one(
    "h3 a"
)["href"]

print("Judul:", title)
print("Harga:", price)
print("Ketersediaan:", availability)
print("Rating:", rating)
print("URL Detail:", detail_url)


Judul: A Light in the Attic
Harga: Â£51.77
Ketersediaan: In stock
Rating: Three
URL Detail: catalogue/a-light-in-the-attic_1000/index.html


##Membersihkan, memeriksa, dan menyimpan data 

In [6]:

records = []

for page in range(1, 6):

    page_url = (
        f"https://books.toscrape.com/catalogue/page-{page}.html"
    )

    page_response = requests.get(
        page_url,
        headers=headers,
        timeout=20
    )

    page_response.raise_for_status()

    page_soup = BeautifulSoup(
        page_response.text,
        "html.parser"
    )

    items = page_soup.select("article.product_pod")

    for item in items:

        records.append({
            "judul": item.select_one("h3 a")["title"],

            "harga": item.select_one(
                ".price_color"
            ).get_text(strip=True),

            "ketersediaan": item.select_one(
                ".availability"
            ).get_text(strip=True),

            "rating": item.select_one(
                "p.star-rating"
            )["class"][1],

            "url_detail": item.select_one(
                "h3 a"
            )["href"],

            "halaman": page
        })

    print(
        f"Halaman {page}: {len(items)} buku berhasil diambil"
    )

    time.sleep(1)

df_books = pd.DataFrame(records)

print("\nTotal Data Buku:", len(df_books))

display(df_books.head())


Halaman 1: 20 buku berhasil diambil
Halaman 2: 20 buku berhasil diambil
Halaman 3: 20 buku berhasil diambil
Halaman 4: 20 buku berhasil diambil
Halaman 5: 20 buku berhasil diambil

Total Data Buku: 100


,judul,harga,ketersediaan,rating,url_detail,halaman
0,A Light in the Attic,Â£51.77,In stock,Three,a-light-in-the-attic_1000/index.html,1
1,Tipping the Velvet,Â£53.74,In stock,One,tipping-the-velvet_999/index.html,1
2,Soumission,Â£50.10,In stock,One,soumission_998/index.html,1
3,Sharp Objects,Â£47.82,In stock,Four,sharp-objects_997/index.html,1
4,Sapiens: A Brief History of Humankind,Â£54.23,In stock,Five,sapiens-a-brief-history-of-humankind_996/index...,1


# Membersihkan harga buku

In [7]:

df_books["harga_numerik"] = (
    df_books["harga"]
    .astype(str)
    .str.replace("Â", "", regex=False)
    .str.replace("£", "", regex=False)
    .str.replace(r"[^0-9.]", "", regex=True)
)

df_books["harga_numerik"] = pd.to_numeric(
    df_books["harga_numerik"],
    errors="coerce"
)

print(df_books[["harga", "harga_numerik"]].head())


     harga  harga_numerik
0  Â£51.77          51.77
1  Â£53.74          53.74
2  Â£50.10          50.10
3  Â£47.82          47.82
4  Â£54.23          54.23


In [8]:
print(df_books["harga_numerik"].describe())


count    100.000000
mean      34.560700
std       14.638531
min       10.160000
25%       19.897500
50%       34.775000
75%       47.967500
max       58.110000
Name: harga_numerik, dtype: float64


# Memeriksa kualitas data buku

In [9]:

print("Jumlah baris dan kolom:")
print(df_books.shape)

print("\nJumlah data kosong:")
print(df_books.isnull().sum())

print("\nJumlah data duplikat:")
print(df_books.duplicated(
    subset=["judul", "url_detail"]
).sum())

print("\nInformasi dataset:")
df_books.info()


Jumlah baris dan kolom:
(100, 7)

Jumlah data kosong:
judul            0
harga            0
ketersediaan     0
rating           0
url_detail       0
halaman          0
harga_numerik    0
dtype: int64

Jumlah data duplikat:
0

Informasi dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   judul          100 non-null    object 
 1   harga          100 non-null    object 
 2   ketersediaan   100 non-null    object 
 3   rating         100 non-null    object 
 4   url_detail     100 non-null    object 
 5   halaman        100 non-null    int64  
 6   harga_numerik  100 non-null    float64
dtypes: float64(1), int64(1), object(5)
memory usage: 5.6+ KB


# Menyimpan CSV Books to Scrape

In [11]:

df_books.to_csv(
    "books_to_scrape_100.csv",
    index=False,
    encoding="utf-8-sig"
)

print("File berhasil disimpan!")


File berhasil disimpan!


#Praktik 2 – Scraping Ulasan Aplikasi Gojek

In [ ]:
## Instalasi dan impor 

In [14]:
%pip install google-play-scraper

from google_play_scraper import app, reviews, Sort
import pandas as pd


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


##Mengambil ulasan aplikasi

In [18]:
app_id = "com.gojek.app"

In [21]:
app_info = app(
    app_id,
    lang="id",
    country="id"
)

print("Nama Aplikasi:", app_info.get("title"))
print("Developer:", app_info.get("developer"))
print("Rating:", app_info.get("score"))
print("Jumlah Rating:", app_info.get("ratings"))
print("Jumlah Ulasan:", app_info.get("reviews"))


Nama Aplikasi: Gojek - Transportasi & Makanan
Developer: PT. GoTo Gojek Tokopedia Tbk
Rating: 4.671551
Jumlah Rating: 6737190
Jumlah Ulasan: 1899106


In [22]:

result, continuation_token = reviews(
    app_id,
    lang="id",
    country="id",
    sort=Sort.MOST_RELEVANT,
    count=100
)

df_reviews = pd.DataFrame(result)

print("Jumlah Ulasan:", len(df_reviews))

display(df_reviews.head())

Jumlah Ulasan: 100


,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,e300b8f6-9c61-4e62-9b72-f2a75336976c,Digwikan Rahmajati,https://play-lh.googleusercontent.com/a-/ALV-U...,"maps driver dan pelanggan tidak sama, maps dri...",2,90,5.76.2,2026-09-14 18:44:02,"Hai, Kak Digwikan. Pastikan kamu sudah memberi...",2026-09-14 19:10:54,5.76.2
1,82f2ece2-9463-4d7a-a9b8-3dda60293f17,Evan Joko Erfianto,https://play-lh.googleusercontent.com/a-/ALV-U...,"Promo'nya skarang makin berkurang, Dibanding y...",2,60,5.75.2,2026-09-11 12:13:57,"Hai Kak Evan, mohon maaf atas kendala yang kam...",2026-09-11 12:41:31,5.75.2
2,3f8b5200-e149-4a23-9f78-f9d5d3f9b5fa,yani c,https://play-lh.googleusercontent.com/a/ACg8oc...,diminta verifikasi email tp email yg tertera s...,1,37,5.74.1,2026-09-02 08:53:50,"Hai Kak Yani, mohon maaf atas ketidaknyamanann...",2026-09-02 10:34:48,5.74.1
3,edbfd9e9-f1a3-4803-99fa-f184e1487fdb,Nico Robin,https://play-lh.googleusercontent.com/a-/ALV-U...,Pelanggan setia gofood tapi suka jengkel masal...,1,42,5.73.2,2026-08-27 12:35:17,"Hai, Kak Nico. Pastikan kamu sudah memberikan ...",2026-08-27 13:06:18,5.73.2
4,2c237e01-60bb-4a5d-afba-4af1e7960b69,Maryama Tri Wulandari,https://play-lh.googleusercontent.com/a-/ALV-U...,"Untuk perjalanan jauh, driver terlalu slow res...",1,32,5.70.1,2026-08-06 05:36:51,"Hai Kak Maryama, mohon maaf atas keterlambatan...",2026-08-06 06:42:53,5.70.1


##Memeriksa dan menyimpan data

In [25]:
print("Jumlah ulasan:", len(df_reviews))

print(df_reviews.columns.tolist())

print(df_reviews.isnull().sum())


df_reviews.to_csv(
    "review_gojek_100.csv",
    index=False,
    encoding="utf-8-sig"
)

print("File berhasil disimpan!")


Jumlah ulasan: 100
['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion']
reviewId                 0
userName                 0
userImage                0
content                  0
score                    0
thumbsUpCount            0
reviewCreatedVersion     0
at                       0
replyContent            21
repliedAt               21
appVersion               0
dtype: int64
File berhasil disimpan!
